In [1]:
import sys
sys.path.insert(1, '/Users/linusrandud/Documents/UoM/ERP/MscDissertation/Deep-Opt')

In [2]:
# %env PYTORCH_ENABLE_MPS_FALLBACK=1

In [3]:
import torch
import wandb
import json
import numpy as np
import matplotlib.pyplot as plt
import random
import uuid
import optuna

In [4]:
from COProblems.MKP import MKP
from COProblems.QUBO import QUBO
from Models.DOAE import DOAE
from OptimAE import OptimAEHandler

In [5]:
def check_constraints(solution, jobs):
    machine_jobs = [[], []]
    makespans = [0, 0]
    infeasible_count = 0
    
    for job_index, job_assignment in enumerate(solution):
        machine = int(job_assignment)
        job_key = job_index + 1
        job = jobs[job_key]
        machine_jobs[machine].append((job_key, job))
    
    for machine, assigned_jobs in enumerate(machine_jobs):
        current_time = 0
        for job_key, job in sorted(assigned_jobs, key=lambda x: (x[1]['deadline'], x[1]['release'])):
            if current_time < job['release']:
                current_time = job['release']
            current_time += job['duration']
            if current_time > job['deadline']:
                infeasible_count += 1
        makespans[machine] = current_time
    
    if infeasible_count > 0:
        return False, infeasible_count, makespans
    return True, infeasible_count, makespans

def find_extreme_indices(fitnesses, mode='high'):
    if mode not in ['high', 'low']:
        raise ValueError("Mode should be either 'high' or 'low'")

    if mode == 'high':
        extreme_value = max(fitnesses)
    else:
        extreme_value = min(fitnesses)

    return [i for i, value in enumerate(fitnesses) if value == extreme_value]

def convert_tensor_to_unique_np_arrays(tensor):
    np_array = tensor.numpy()
    np_array[np_array == -1] = 0
    unique_arrays = set()
    unique_np_arrays = []
    
    for arr in np_array:
        arr_tuple = tuple(arr)
        if arr_tuple not in unique_arrays:
            unique_arrays.add(arr_tuple)
            unique_np_arrays.append(arr)
    
    unique_np_array = np.array(unique_np_arrays)
    return unique_np_array

def get_solutions(population, fitnesses, mode='low'):
    return convert_tensor_to_unique_np_arrays(population[find_extreme_indices(fitnesses, mode)])

def load_jobs_from_json(file_path):
    with open(file_path, 'r') as f:
        jobs = json.load(f)
    
    # Convert keys to integers
    jobs = {int(k): v for k, v in jobs.items()}
    
    return jobs

In [6]:
# New parameter sets
# datasets = ['ssjsp_4', 'ssjsp_8', 'ssjsp_16', 'ssjsp_20']
dataset = 'ssjsp_100_s60'
constraint_methods = [None, 'binary', 'lagrangian']
pop_size_options = [1000]

# Parameters
base_params = {
    'change_tolerance': 50,
    'problem_size': None,  # Will be set for each dataset
    'pop_size': None,
    'dropout_prob': 0.2,
    'l1_coef': 0.0001,
    'l2_coef': 0.0001,
    'learning_rate': 0.002,
    'max_depth': 6,
    'compression_ratio': 0.8,
    'problem_instance_id': 0,
    'deepest_only': False,
    'encode': True,
    'repair_solutions': True,
    'patience': 5,  # Number of iterations to wait
    'delta_mean_population': 0.1,  # Threshold for mean population change
    'check_constraints': None,
    'penalty_mult': 5
}

# Number of iterations for stability
num_iterations = 3
device = torch.device("cpu")
problem_type = 'QUBO'
use_wandb = False

In [7]:
def objective(trial):
    # Hyperparameters to optimize
    change_tolerance = trial.suggest_int("change_tolerance", 10, 100, step=10)
    pop_size = trial.suggest_int("pop_size", 500, 2000, step=100)
    max_depth = trial.suggest_int("max_depth", 2, 10, step=2)
    constraint_method = 'lagrangian'

    # Parameters for Deep Optimization
    base_params = {
        'dropout_prob': 0.2,
        'l1_coef': 0.0001,
        'l2_coef': 0.0001,
        'learning_rate': 0.002,
        'compression_ratio': 0.8,
        'problem_instance_id': 0,
        'deepest_only': False,
        'encode': True,
        'repair_solutions': True,
        'patience': 5,
        'delta_mean_population': 0.1,
        'check_constraints': constraint_method,
        'penalty_mult': 5
    }

    # Update with trial suggestions
    base_params.update({
        'change_tolerance': change_tolerance,
        'pop_size': pop_size,
        'max_depth': max_depth
    })
    
    # Load jobs
    json_file = f'../data_gecco/ssjsp/{dataset}.json'
    jobs = load_jobs_from_json(json_file)

    # Set problem size based on the dataset
    file_paths = [f'../data_gecco/qubo/{dataset}.txt']
    problem_size = len(jobs)
    base_params['problem_size'] = problem_size
    base_params['pop_size'] = pop_size
    base_params['constraint_method'] = constraint_method

    print(f"\nStarting experiments for dataset {dataset} with population size {pop_size} and constraint method {constraint_method}")

    # Define a group name for the experiment
    group_name = f"{dataset}_popsize_{pop_size}_constraint_{constraint_method}"


    iteration = 0
    unique_id = uuid.uuid4().hex  # Generate a unique ID for the pair
    print(f"Iteration {iteration + 1}/{num_iterations} for {dataset}, constraint method: {constraint_method}, population size: {pop_size}")

    if use_wandb:
        wandb.init(
            project="GECCO DO with Constraints ",
            group=group_name,  # Grouping experiments
            tags=[problem_type, f"constraint_method={constraint_method}", f"id={unique_id}"],
            name=f"{dataset}_run_{iteration + 1}_{constraint_method}"
        )
        wandb.config.update(base_params)
        wandb.log_artifact(json_file, type='dataset')

    if problem_type == 'QUBO':
        problem = QUBO(file_paths[0], base_params['problem_instance_id'], device)
        if base_params['constraint_method']:
            problem.jobs = jobs
    elif problem_type == 'MKP':
        problem = MKP(file_paths[0], file_paths[1], base_params['problem_instance_id'], device)
    else:
        raise ValueError("Unsupported problem type")

    model = DOAE(base_params['problem_size'], base_params['dropout_prob'], device)
    handler = OptimAEHandler(model, problem, device)

    population, fitnesses = handler.generate_population(base_params['pop_size'], base_params['constraint_method'], base_params['penalty_mult'])
    population, fitnesses, _, _ = handler.hilldescent(population, fitnesses, base_params['change_tolerance'], base_params['constraint_method'], base_params['penalty_mult'])
    handler.print_statistics_min(fitnesses)

    total_eval = 0
    depth = 0

    mean_fitnesses = []
    min_max_fitnesses = []
    total_evaluations = []
    mean_fitness_changes = []

    while True:
        if depth < base_params['max_depth']:
            hidden_size = round(base_params['problem_size'] * base_params['compression_ratio'])
            model.transition(hidden_size)
            depth += 1
            optimizer = torch.optim.Adam(model.parameters(), lr=base_params['learning_rate'], weight_decay=base_params['l2_coef'])
        
        handler.learn_from_population(population, optimizer, l1_coef=base_params['l1_coef'], batch_size=base_params['pop_size'])
        
        population, fitnesses, evaluations, done = handler.optimise_solutions_min(
            population, fitnesses, base_params['change_tolerance'], encode=base_params['encode'], repair_solutions=base_params['repair_solutions'], deepest_only=base_params['deepest_only'], 
            check_constraints=base_params['constraint_method'], penalty_mult=base_params['penalty_mult']
        )
        handler.print_statistics_min(fitnesses)

        mean_fitness = fitnesses.mean().item()
        min_max_fitness = fitnesses.min().item()
        total_eval += evaluations

        mean_fitnesses.append(mean_fitness)
        min_max_fitnesses.append(min_max_fitness)
        total_evaluations.append(total_eval)

        print(f"Iteration {iteration + 1}, Depth {depth}, Evaluations: {total_eval}, Mean Fitness: {mean_fitness:.4f}, Min Fitness: {min_max_fitness:.4f}")

        if use_wandb:
            wandb.log({
                "mean_fitness": mean_fitness,
                "min_max_fitness": min_max_fitness,
                "total_eval": total_eval,
                "depth": depth,
                "current_iteration": iteration + 1,
                "population_size": pop_size,
                "constraint_method": constraint_method,
                "dataset": dataset,
            })

        if len(mean_fitnesses) > 1:
            mean_fitness_change = abs(mean_fitnesses[-1] - mean_fitnesses[-2])
            mean_fitness_changes.append(mean_fitness_change)
            
            if len(mean_fitness_changes) >= base_params['patience']:
                recent_changes = mean_fitness_changes[-base_params['patience']:]
                if all(change < base_params['delta_mean_population'] for change in recent_changes):
                    break
        
        if done:
            break

    plt.figure(figsize=(10, 6))
    plt.plot(total_evaluations, mean_fitnesses, label='Mean Fitness')
    plt.plot(total_evaluations, min_max_fitnesses, label='Max Fitness')
    plt.axhline(y=problem.max_fitness, color='r', linestyle='--', label='Max Possible Fitness')
    plt.xlabel('Evaluations')
    plt.ylabel('Fitness')
    plt.title(f'Mean and Max Fitness over Evaluations (Iteration {iteration + 1}, Constraint Method: {constraint_method})')
    plt.legend()

    if use_wandb:
        wandb.log({"fitness_plot": wandb.Image(plt)})

    solutions = get_solutions(population, fitnesses, mode='low')
    total_feasible_solutions = 0
    total_infeasible_solutions = 0
    logs = []

    for i, solution in enumerate(solutions):
        feasible, infeasible_jobs, makespans = check_constraints(solution, jobs)
        if feasible:
            total_feasible_solutions += 1
        else:
            total_infeasible_solutions += 1
        
        logs.append({
            'solution_number': i + 1,
            'feasible': feasible,
            'infeasible_jobs': infeasible_jobs,
            'makespans': makespans
        })

    print(f"Completed Iteration {iteration + 1} for dataset {dataset} with constraint method {constraint_method}.")
    print(f'Total feasible solutions: {total_feasible_solutions}')
    print(f'Total infeasible solutions: {total_infeasible_solutions}')

    if use_wandb:
        wandb.log({
            'solutions': logs,
            'total_feasible_solutions_count': total_feasible_solutions,
            'total_infeasible_solutions': total_infeasible_solutions
        })

    if use_wandb:
        wandb.finish()
    
    return mean_fitness, min_max_fitness, feasible

In [8]:
# Create a study and optimize
study = optuna.create_study(directions=["minimize", "minimize", "maximize"])
study.optimize(objective, n_trials=10)

[I 2025-04-18 15:18:15,949] A new study created in memory with name: no-name-86818cb0-93b6-4e37-bd8c-a4c564c0379b



Starting experiments for dataset ssjsp_100_s60 with population size 2000 and constraint method lagrangian
Iteration 1/3 for ssjsp_100_s60, constraint method: lagrangian, population size: 2000
Instance has been loaded
Min pop fitness: -46375408.0, Mean pop fitness : -46368460.0
Min pop fitness: -46375632.0, Mean pop fitness : -46371292.0
Iteration 1, Depth 1, Evaluations: 31229, Mean Fitness: -46371292.0000, Min Fitness: -46375632.0000
Min pop fitness: -46375760.0, Mean pop fitness : -46373048.0
Iteration 1, Depth 2, Evaluations: 86213, Mean Fitness: -46373048.0000, Min Fitness: -46375760.0000
Min pop fitness: -46375760.0, Mean pop fitness : -46373792.0
Iteration 1, Depth 3, Evaluations: 161450, Mean Fitness: -46373792.0000, Min Fitness: -46375760.0000
Min pop fitness: -46375848.0, Mean pop fitness : -46374216.0
Iteration 1, Depth 4, Evaluations: 257623, Mean Fitness: -46374216.0000, Min Fitness: -46375848.0000
Min pop fitness: -46375848.0, Mean pop fitness : -46374500.0
Iteration 1, D

[W 2025-04-18 15:20:20,206] Trial 0 failed with parameters: {'change_tolerance': 10, 'pop_size': 2000, 'max_depth': 6} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/linusrandud/anaconda3/envs/myenv_3.9/lib/python3.9/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/fh/7sz134vx5jq6_78w16smcs3h0000gn/T/ipykernel_22017/1656732692.py", line 96, in objective
    population, fitnesses, evaluations, done = handler.optimise_solutions_min(
  File "/Users/linusrandud/anaconda3/envs/myenv_3.9/lib/python3.9/site-packages/torch/autograd/grad_mode.py", line 27, in decorate_context
    return func(*args, **kwargs)
  File "/Users/linusrandud/Documents/UoM/ERP/MscDissertation/Deep-Opt/OptimAE.py", line 160, in optimise_solutions_min
    evaluations += self.assess_changes_descent(solutions, fitnesses, new_solutions,
  File "/Users/linusrandud/anaconda3/envs/myenv_3.9/lib/python

KeyboardInterrupt: 

In [ ]:
study.best_trials

[FrozenTrial(number=2, state=TrialState.COMPLETE, values=[-46375908.0, -46376104.0, 1.0], datetime_start=datetime.datetime(2025, 1, 13, 4, 1, 26, 269677), datetime_complete=datetime.datetime(2025, 1, 13, 5, 42, 27, 282632), params={'change_tolerance': 90, 'pop_size': 1100, 'max_depth': 8}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributions={'change_tolerance': IntDistribution(high=100, log=False, low=10, step=10), 'pop_size': IntDistribution(high=2000, log=False, low=500, step=100), 'max_depth': IntDistribution(high=10, log=False, low=2, step=2)}, trial_id=2, value=None),
 FrozenTrial(number=7, state=TrialState.COMPLETE, values=[-46375932.0, -46376100.0, 1.0], datetime_start=datetime.datetime(2025, 1, 13, 8, 39, 46, 796340), datetime_complete=datetime.datetime(2025, 1, 13, 11, 52, 3, 86698), params={'change_tolerance': 100, 'pop_size': 1800, 'max_depth': 10}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributio

In [ ]:
from optuna.visualization import plot_slice

In [ ]:
fig = plot_slice(study, target=lambda t: t.values[0], target_name='Mean Fitness')

In [ ]:
fig.update_layout(
    title=None,  # Remove global title
)

# Update the axes for all subplots
fig.update_xaxes(
    title=dict(font=dict(size=16, color="black", family="Arial Black")),  # Make X-axis title bold
    tickfont=dict(size=14)  # Adjust X-axis tick labels
)
fig.update_yaxes(
    title=dict(font=dict(size=16, color="black", family="Arial Black")),  # Make Y-axis title bold
    tickfont=dict(size=14)  # Adjust Y-axis tick labels
)


In [ ]:
import plotly.io as pio
pio.write_image(fig, '../figs/sliceplot.png', width=800, height=400)

In [ ]:
from optuna.visualization import plot_parallel_coordinate

# Parallel coordinate plot to explore parameter interactions
fig = plot_parallel_coordinate(study, target=lambda t: t.values[0])
fig.show()

In [ ]:
import pandas as pd

# Export trial data
df = study.trials_dataframe()
print(df.head())

   number    values_0    values_1  values_2             datetime_start  \
0       0 -46375876.0 -46376104.0       1.0 2025-01-13 02:01:20.823165   
1       1 -46375728.0 -46376088.0       0.0 2025-01-13 03:16:52.374406   
2       2 -46375908.0 -46376104.0       1.0 2025-01-13 04:01:26.269677   
3       3 -46375720.0 -46376084.0       0.0 2025-01-13 05:42:27.283466   
4       4 -46375832.0 -46376080.0       0.0 2025-01-13 05:52:54.869441   

           datetime_complete               duration  params_change_tolerance  \
0 2025-01-13 03:16:52.371798 0 days 01:15:31.548633                       60   
1 2025-01-13 04:01:26.268613 0 days 00:44:33.894207                       20   
2 2025-01-13 05:42:27.282632 0 days 01:41:01.012955                       90   
3 2025-01-13 05:52:54.868978 0 days 00:10:27.585512                       70   
4 2025-01-13 06:21:02.130997 0 days 00:28:07.261556                       50   

   params_max_depth  params_pop_size  system_attrs_nsga2:generation     st

In [ ]:
df[['values_0',	'values_1',	'values_2', 'params_change_tolerance',	'params_max_depth',	'params_pop_size']].corr()

,values_0,values_1,values_2,params_change_tolerance,params_max_depth,params_pop_size
values_0,1.000000,0.664882,-0.692791,-0.784571,-0.877246,0.151285
values_1,0.664882,1.000000,-0.699854,-0.527046,-0.544389,-0.283224
values_2,-0.692791,-0.699854,1.000000,0.726184,0.620490,0.061361
params_change_tolerance,-0.784571,-0.527046,0.726184,1.000000,0.548544,-0.280088
params_max_depth,-0.877246,-0.544389,0.620490,0.548544,1.000000,0.072364
params_pop_size,0.151285,-0.283224,0.061361,-0.280088,0.072364,1.000000


In [ ]:
import pickle

# Save the study to a file
with open("optuna_study.pkl", "wb") as f:
    pickle.dump(study, f)
print("Study saved to 'optuna_study.pkl'")


Study saved to 'optuna_study.pkl'


In [ ]:
import pickle

# Load the study from a file
with open("optuna_study.pkl", "rb") as f:
    loaded_study = pickle.load(f)

print("Study loaded successfully")
# print("Best trial:", loaded_study.best_trial)


Study loaded successfully


In [ ]:
study = loaded_study

In [ ]:
study.set_metric_names(["Mean Fitness", "Min Fitness", "# feasible solutions"])
fig = optuna.visualization.plot_param_importances(study)

/var/folders/fh/7sz134vx5jq6_78w16smcs3h0000gn/T/ipykernel_8123/1929298960.py:1: ExperimentalWarning:

set_metric_names is experimental (supported from v3.2.0). The interface can change in the future.



In [ ]:
fig.update_layout(
    title=None,
    yaxis=dict(
        tickangle=-90,  # Rotates tick labels on the Y-axis
        tickfont=dict(size=14),
        title=dict(
            font=dict(size=16, family="Arial Black", color="black")
        )
    ),
    xaxis=dict(
        tickfont=dict(size=14),
        title=dict(font=dict(size=16, family="Helvetica Bold, sans-serif", color="black"))
    ),
    legend=dict(
        font=dict(size=15)
    )
)

In [ ]:
import plotly.io as pio
pio.write_image(fig, '../figs/Hyperparameter Importances.png', width=800, height=450)

In [ ]:
optuna.visualization.plot_optimization_history(study, target=lambda t: t.values[1])

/Users/linusrandud/anaconda3/envs/myenv_3.9/lib/python3.9/site-packages/optuna/visualization/_utils.py:67: UserWarning:

`target` is specified, but `target_name` is the default value, 'Objective Value'.

